Постановка задачи

Найти вещественный корень уравнения:

$$ f(x) = x^m - a = 0 $$

где $a > 0$, $m \in \mathbb{N}$, $m \geq 2$.

Метод Чебышёва

Пусть $f(x)$ — достаточно гладкая функция, и $x_*$ — её корень $(f(x_*) = 0)$.  
Разложим обратную функцию $x = F(y)$ в ряд Тейлора в окрестности $y_0 = f(x_0)$:

$$ x = F(y) = F(y_0) + F'(y_0)(y - y_0) + \frac{F''(y_0)}{2!}(y - y_0)^2 + \dots + \frac{F^{(k)}(y_0)}{k!}(y - y_0)^k + \dots $$

Здесь $F$ — функция, обратная к $f$, т.е. $F(f(x)) = x$.  
В частности, для задачи $f(x) = x^m - a$ имеем:

$$ F(y) = (y + a)^{1/m} $$

Полагая $y = 0$ (так как ищем корень $f(x_*) = 0$), получаем итерационную формулу Чебышёва порядка $p$:

$$ x_{n+1} = x_n + \sum_{k=1}^{p} \frac{F^{(k)}(y_n)}{k!} (-y_n)^k $$

где $y_n = f(x_n)$.

Вычисление производных $F^{(k)}$

Для $F(y) = (y + a)^{1/m}$ производные вычисляются аналитически:

$$ F'(y) = \frac{1}{m}(y + a)^{1/m - 1} $$

$$ F''(y) = \frac{1}{m}\left(\frac{1}{m} - 1\right)(y + a)^{1/m - 2} $$

и в общем виде:

$$ F^{(k)}(y) = \left(\frac{1}{m}\right)\left(\frac{1}{m} - 1\right)\dots\left(\frac{1}{m} - k + 1\right)(y + a)^{1/m - k} $$

В коде производные вычисляются символьно с использованием библиотеки SymPy.

Алгоритм


## Алгоритм

1. Задать начальное приближение $x_0$, порядок метода $p$, точность $\varepsilon$, максимальное число итераций $N_{\max}$.

2. Для $n = 0, 1, \dots, N_{\max}$:
   1. Вычислить $y_n = x_n^m - a$.
   2. Вычислить производные $F^{(k)}(y_n)$ для $k = 1, \dots, p$.
   3. Найти новое приближение:
   
   $$ x_{n+1} = x_n + \sum_{k=1}^{p} \frac{F^{(k)}(y_n)}{k!} (-y_n)^k $$
   
   4. Если $|x_{n+1} - x_n| < \varepsilon$, остановиться.

3. Вывести приближённое решение $x_{n+1}$ и историю итераций.


<!-- Исследование сходимости

Пусть $x_*$ — точный корень $(x_* = a^{1/m})$.  
Абсолютная ошибка на $n$-й итерации:

$$ e_n = |x_n - x_*| $$

Для метода Чебышёва порядка $p$ скорость сходимости вблизи корня оценивается как:

$$ e_{n+1} = O(e_n^{p+1}) $$

при условии достаточной гладкости функции $F(y)$ и близости начального приближения к корню. -->

In [43]:
import sympy as sp
import math
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np


x_sym = sp.symbols('x')

# def F(x_val, a, m):
#     # return (x_val + a)**(1/m)
#     return math.log(x_val+a)

# def nth_derivative(n, a, m):
#     F_sym = (x_sym + a)**(1/m)
#     F_n_sym = sp.diff(F_sym, x_sym, n)
#     return sp.lambdify(x_sym, F_n_sym, 'math')

def cheb(a, m, x0, order=2, max_iter=100, tol=1e-10):
    x = x0
    # y = np.exp(x)-a

    history = [x]  
    
    for iter in range(max_iter):
        # y = x**m - a
        y=np.exp(x) - a
        # x_new = x
        
        # for k in range(1, order+1):
        #     deriv = nth_derivative(k, a, m)  
        #     x_new += deriv(y) / math.factorial(k) * (-y)**k
        # x_new = np.log(y+a) - y/(y+a) - (y**2)/(2*(y+a)**2) - (y**3)/(3*(y+a)**3)
        x_new = np.log(y+a) - y/(y+a) - 0*(y**2)/(2*(y+a)**2) - (0*y**3)/(3*(y+a)**3)


        
        history.append(x_new)
        # print(x_new)
        
        
        if abs(x_new - x) < tol:
            print(f"Метод сошелся за {iter + 1} итераций для order = {order}")
            # print(f'x**m - a = {x_new**m - a}')
            print()
            return x, history


        x = x_new
    print(f"Метод не сошелся за {max_iter} итераций для order = {order}")
    return x, history

# Параметры
a = 3
m = 1
x0 = 1
# orders = [1,2,3,4,5]
orders = [3]


# exact_root = a**(1/m)
exact_root = np.log(a)
fig = go.Figure()

for order in orders:
    root, history = cheb(a, m, x0, order=order)
    errors = [(x - exact_root) for x in history]
    
    fig.add_trace(go.Scatter(
        x=list(range(len(errors))),
        y=np.abs(errors),
        mode='lines+markers',
        name=f'{order} члена'
    ))

# fig.update_layout(
#     title=f'График ошибки приближений для корня {m}-й степени из {a}',
#     xaxis_title='Итерация',
#     yaxis_title='Абсолютная ошибка',
# )

fig.show()

Метод сошелся за 4 итераций для order = 3



In [40]:
errors

[np.float64(-0.09861228866810978),
 np.float64(2.6640158041990603e-05),
 np.float64(0.0),
 np.float64(-2.220446049250313e-16)]

In [42]:
errors

[np.float64(-0.09861228866810978),
 np.float64(-0.0003444162042129939),
 np.float64(-1.362199242294082e-11),
 np.float64(-2.220446049250313e-16)]

In [44]:
errors

[np.float64(-0.09861228866810978),
 np.float64(0.00502603484621722),
 np.float64(1.260937923275307e-05),
 np.float64(7.949774172288926e-11),
 np.float64(-2.220446049250313e-16)]